In [ ]:
import os
import asyncio
from typing import List

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.runtime import SingleThreadedAgentRuntime
from autogen_agentchat.messages import Message
from autogen_agentchat.types import AgentId
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")
print("API key loaded:", bool(api_key))

In [ ]:
model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini"
)

### MCP-Style Tools

In [ ]:
def get_trending_topics(topic):
    return [
        f"Latest innovations in {topic}",
        f"How AI is transforming {topic}",
        f"Future of {topic} with AI agents"
    ]


def generate_posting_time():
    return "Best time to post: 9 AM - 11 AM"

### Create Agent
Trend research Agent

In [ ]:
trend_agent = AssistantAgent(
    name="trend_agent",
    model_client=model_client,
    system_message="""
You are a social media trend analyst.
Identify engaging angles for the topic and suggest viral ideas.
"""
)

### Content Planner Agent 

In [ ]:
planner_agent = AssistantAgent(
    name="planner_agent",
    model_client=model_client,
    system_message="""
Create a structure for a social media post.

Format:
Hook
Key Points
Call to Action
"""
)

### Content Writer Agent 

In [ ]:
writer_agent = AssistantAgent(
    name="writer_agent",
    model_client=model_client,
    system_message="""
Write a highly engaging social media post.
Make it professional and exciting.
"""
)

### Platform Optimizer Agent

In [ ]:
optimizer_agent = AssistantAgent(
    name="optimizer_agent",
    model_client=model_client,
    system_message="""
Optimize the content specifically for LinkedIn.
Make it readable and impactful.
"""
)

### Hastage Generator Agent

In [ ]:
hashtag_agent = AssistantAgent(
    name="hashtag_agent",
    model_client=model_client,
    system_message="""
Generate trending hashtags relevant to the post.
"""
)

### Setup Runner

In [ ]:
runtime = SingleThreadedAgentRuntime()

runtime.register_agent(trend_agent)
runtime.register_agent(planner_agent)
runtime.register_agent(writer_agent)
runtime.register_agent(optimizer_agent)
runtime.register_agent(hashtag_agent)

runtime.start()

### User Input 

In [ ]:
topic = input("Enter topic: ")
platform = input("Platform (LinkedIn/Twitter): ")

### Get trending topic (Tools Useage)

In [ ]:
trends = get_trending_topics(topic)
print("Trending Angles:", trends)

### Treand Analysis

In [ ]:
response1 = await runtime.send_message(
    Message(f"Topic: {topic}\nTrends: {trends}"),
    AgentId("trend_agent", "default")
)

print(response1.content)

### Plan Content

In [ ]:
response2 = await runtime.send_message(
    Message(response1.content),
    AgentId("planner_agent", "default")
)

print(response2.content)

### Write POST

In [ ]:
response3 = await runtime.send_message(
    Message(response2.content),
    AgentId("writer_agent", "default")
)

print(response3.content)

### Optimize for platform

In [ ]:
response4 = await runtime.send_message(
    Message(f"Platform: {platform}\n{response3.content}"),
    AgentId("optimizer_agent", "default")
)

print(response4.content)

### Generate Hastages

In [ ]:
response5 = await runtime.send_message(
    Message(response4.content),
    AgentId("hashtag_agent", "default")
)

print("\nFinal Post:\n")
print(response4.content)
print("\nHashtags:\n")
print(response5.content)